# Task 01: Head Direction Detection

Detect which way the student's head is pointing:
- DOWN/FORWARD = looking at paper (normal)
- LEFT/RIGHT = looking at neighbor (suspicious)
- UP = looking away (suspicious)

Uses **MediaPipe Face Landmarker** — pre-trained, zero data needed.

**Press 'q' to quit.**

## Cell 1: Import and Setup

Load MediaPipe Face Landmarker. This model tracks 478 points on your face.

In [1]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
import numpy as np
import time

# Initialize MediaPipe Face Landmarker (new API)
model_path = 'face_landmarker.task'

base_options = mp_python.BaseOptions(model_asset_path=model_path)
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    running_mode=vision.RunningMode.IMAGE
)
landmarker = vision.FaceLandmarker.create_from_options(options)

print("MediaPipe Face Landmarker loaded!")
print("Tracks 478 points on your face — zero training needed.")

MediaPipe Face Landmarker loaded!
Tracks 478 points on your face — zero training needed.


## Cell 2: Configuration

Change these numbers and re-run to see the effect.

In [2]:
# =============================================
# CONFIGURATION - Tune these numbers!
# =============================================

TURN_THRESHOLD = 0.15        # How much head turn counts as 'turned' (0.1=strict, 0.2=loose)
SUSPICIOUS_DURATION = 3.0    # Seconds of continuous turning = suspicious
UP_THRESHOLD = 0.1           # Head tilt up threshold

print("Configuration:")
print(f"  Turn threshold:     {TURN_THRESHOLD}")
print(f"  Suspicious after:   {SUSPICIOUS_DURATION} seconds")
print(f"  Up threshold:       {UP_THRESHOLD}")

Configuration:
  Turn threshold:     0.15
  Suspicious after:   3.0 seconds
  Up threshold:       0.1


## Cell 3: Head Direction Function

Uses nose tip + eye positions to figure out which way the head is pointing.

In [3]:
def get_head_direction(face_landmarks, frame_width, frame_height):
    # Key landmarks
    nose = face_landmarks[4]           # Nose tip
    left_eye = face_landmarks[133]     # Left eye inner corner
    right_eye = face_landmarks[362]    # Right eye inner corner
    
    # Convert to pixel coordinates
    nose_x = nose.x * frame_width
    nose_y = nose.y * frame_height
    left_eye_x = left_eye.x * frame_width
    right_eye_x = right_eye.x * frame_width
    eye_center_x = (left_eye_x + right_eye_x) / 2
    eye_center_y = (left_eye.y + right_eye.y) / 2 * frame_height
    
    eye_distance = abs(right_eye_x - left_eye_x)
    if eye_distance == 0:
        return "UNKNOWN", 0, 0
    
    # How far nose is from center of eyes
    horizontal_ratio = (nose_x - eye_center_x) / eye_distance
    vertical_ratio = (nose_y - eye_center_y) / eye_distance
    
    # Determine direction
    if vertical_ratio < -UP_THRESHOLD:
        direction = "UP"
    elif horizontal_ratio < -TURN_THRESHOLD:
        direction = "LEFT"
    elif horizontal_ratio > TURN_THRESHOLD:
        direction = "RIGHT"
    else:
        direction = "FORWARD"
    
    return direction, horizontal_ratio, vertical_ratio

print("Head direction function ready!")

Head direction function ready!


## Cell 4: Live Detection

- GREEN bar = looking at paper (normal)
- YELLOW bar = turned slightly (might be stretching)
- RED bar = suspicious! Looking away too long

**Press 'q' to quit.**

In [4]:
camera = cv2.VideoCapture(0)
camera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not camera.isOpened():
    print("ERROR: Cannot open webcam!")
else:
    print("Head Direction Detection running!")
    print("Turn your head left/right/up to test. Press 'q' to quit.")
    
    turn_start_time = None
    alert_count = 0
    frame_count = 0
    start_time = time.time()
    
    while True:
        success, frame = camera.read()
        if not success:
            break
        
        # FLIP the frame horizontally (mirror effect)
        # Without this: your left shows as RIGHT on screen (confusing)
        # With this: your left = LEFT on screen (matches your perspective)
        frame = cv2.flip(frame, 1)
        
        frame_count += 1
        height, width = frame.shape[:2]
        
        # Convert to RGB and MediaPipe Image
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        # Detect face landmarks
        results = landmarker.detect(mp_image)
        
        if results.face_landmarks and len(results.face_landmarks) > 0:
            face_lms = results.face_landmarks[0]
            
            # Draw key face points
            for idx in [4, 133, 362, 1, 168]:
                x = int(face_lms[idx].x * width)
                y = int(face_lms[idx].y * height)
                cv2.circle(frame, (x, y), 3, (0, 255, 255), -1)
            
            # Get direction
            direction, h_ratio, v_ratio = get_head_direction(face_lms, width, height)
            
            if direction == "FORWARD":
                color = (0, 200, 0)
                status = "NORMAL - Looking at paper"
                turn_start_time = None
            else:
                if turn_start_time is None:
                    turn_start_time = time.time()
                
                turn_duration = time.time() - turn_start_time
                
                if turn_duration < SUSPICIOUS_DURATION:
                    color = (0, 200, 255)
                    status = f"Looking {direction} ({turn_duration:.1f}s)"
                else:
                    color = (0, 0, 255)
                    status = f"SUSPICIOUS! Looking {direction} ({turn_duration:.1f}s)"
                    if turn_duration < SUSPICIOUS_DURATION + 0.1:
                        alert_count += 1
                        print(f"  Alert #{alert_count}: Looking {direction} for {SUSPICIOUS_DURATION}+ sec")
            
            # Status bar
            cv2.rectangle(frame, (0, 0), (width, 50), color, -1)
            cv2.putText(frame, status, (10, 35),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
            
            # Direction text at bottom
            labels = {"FORWARD": "[  DOWN  ]", "LEFT": "[ << LEFT ]",
                      "RIGHT": "[ RIGHT >> ]", "UP": "[   UP   ]"}
            cv2.putText(frame, labels.get(direction, ""), (width//2 - 80, height - 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
            
            # Debug ratios
            cv2.putText(frame, f"H:{h_ratio:.2f} V:{v_ratio:.2f}", (10, height-10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200,200,200), 1)
        else:
            cv2.rectangle(frame, (0, 0), (width, 50), (100, 100, 100), -1)
            cv2.putText(frame, "No face detected", (10, 35),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        # FPS
        elapsed = time.time() - start_time
        fps = frame_count / elapsed if elapsed > 0 else 0
        cv2.putText(frame, f"FPS:{fps:.0f} Alerts:{alert_count}", (10, height-30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200,200,200), 1)
        
        cv2.imshow('ExamGuard - Head Direction (q to quit)', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    camera.release()
    cv2.destroyAllWindows()
    print(f"Done! {elapsed:.0f}s, {fps:.0f} FPS, {alert_count} alerts")

Head Direction Detection running!
Turn your head left/right/up to test. Press 'q' to quit.
  Alert #1: Looking LEFT for 3.0+ sec
  Alert #2: Looking LEFT for 3.0+ sec
  Alert #3: Looking LEFT for 3.0+ sec
Done! 131s, 29 FPS, 3 alerts
